In [0]:
CREATE TABLE IF NOT EXISTS workspace.myra_invest.gold_stock_recommendation_engine (

    symbol STRING,
    companyName STRING,

    tradeDate DATE,

    close DOUBLE,
    movingAvg5 DOUBLE,
    movingAvg20 DOUBLE,
    dailyReturnPct DOUBLE,

    newsStrength DOUBLE,
    averageSentimentScore DOUBLE,

    technicalScore DOUBLE,
    newsScore DOUBLE,
    finalScore DOUBLE,

    recommendation STRING,

    ingestionDate DATE

)
USING DELTA;

SHOW TABLES IN workspace.myra_invest;

In [0]:
%python
from pyspark.sql.functions import *
from pyspark.sql.window import Window

bronze_df = spark.table("workspace.myra_invest.bronze_stock_prices")

print("Bronze records:", bronze_df.count())
display(bronze_df.limit(5))

In [0]:
%python
window_spec = Window.partitionBy("symbol").orderBy("tradeDate")

window5 = window_spec.rowsBetween(-4, 0)

window20 = window_spec.rowsBetween(-19, 0)

In [0]:
%python
silver_df = (
    bronze_df
    .withColumn(
        "dailyReturnPct",
        round(
            when(col("open") != 0,
                 ((col("close") - col("open")) / col("open")) * 100
            ).otherwise(None),
            2
        )
    )
    .withColumn(
        "movingAvg5",
        round(avg("close").over(window5), 2)
    )
    .withColumn(
        "movingAvg20",
        round(avg("close").over(window20), 2)
    )
)

In [0]:
%python
display(
    silver_df.select(
        "symbol",
        "tradeDate",
        "close",
        "movingAvg5",
        "movingAvg20",
        "dailyReturnPct"
    )
)

In [0]:
%python
(
    silver_df.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .format("delta")
        .saveAsTable("workspace.myra_invest.silver_stock_prices")
)

print("✅ Silver Stock Prices rebuilt successfully.")

In [0]:
DESCRIBE TABLE workspace.myra_invest.silver_stock_prices;

SELECT
    symbol,
    tradeDate,
    close,
    movingAvg5,
    movingAvg20,
    dailyReturnPct
FROM workspace.myra_invest.silver_stock_prices
ORDER BY tradeDate DESC
LIMIT 10;

In [0]:
%python
from pyspark.sql.functions import *

technical_df = spark.table(
    "workspace.myra_invest.gold_stock_recommendations"
)

print("Technical records:", technical_df.count())

display(technical_df.limit(5))

In [0]:
%python
news_df = spark.table(
    "workspace.myra_invest.gold_stock_news_score"
)

print("News records:", news_df.count())

display(news_df.limit(5))

In [0]:
%python

recommendation_df = (
    technical_df.alias("tech")
    .join(
        news_df.alias("news"),
        on=["symbol","companyName"],
        how="left"
    )
    .select(
        col("symbol"),
        col("companyName"),

        col("tradeDate"),

        col("close"),
        col("movingAvg5"),
        col("movingAvg20"),
        col("dailyReturnPct"),

        coalesce(col("newsStrength"), lit(0.0)).alias("newsStrength"),

        coalesce(
            col("averageSentimentScore"),
            lit(0.0)
        ).alias("averageSentimentScore"),

        col("ingestionDate")
    )
)

display(recommendation_df.limit(10))

In [0]:
%python
print("Recommendation dataset:", recommendation_df.count())

recommendation_df.filter(
    col("newsStrength") == 0
).select(
    "symbol",
    "companyName"
).show()